First Pipeline to run binding energy


In [2]:
!uv pip install fairchem-core

Using Python 3.13.11 environment at: C:\Users\miaom\miniconda3\envs\cms
Resolved 91 packages in 4.26s
 Downloaded psycopg2-binary
 Downloaded cryptography
 Downloaded grpcio
 Downloaded tensorboard
 Downloaded numpy
 Downloaded wandb
Prepared 31 packages in 11.93s
Uninstalled 1 package in 320ms
Installed 31 packages in 3.16s
 + absl-py==2.4.0
 + ase-db-backends==0.11.0
 + cloudpickle==3.1.2
 + clusterscope==0.0.18
 + cryptography==46.0.6
 + e3nn==0.6.0
 + fairchem-core==2.12.0
 + gitdb==4.0.12
 + gitpython==3.1.46
 + grpcio==1.80.0
 + hydra-core==1.3.2
 + lmdb==1.6.2
 + markdown==3.10.2
 + mypy-extensions==1.1.0
 - numpy==2.4.1
 + numpy==2.2.6
 + opt-einsum==3.4.0
 + opt-einsum-fx==0.1.4
 + protobuf==6.33.6
 + psycopg2-binary==2.9.11
 + pymysql==1.1.2
 + pyre-extensions==0.0.32
 + sentry-sdk==2.57.0
 + smmap==5.0.3
 + submitit==1.5.4
 + tensorboard==2.20.0
 + tensorboard-data-server==0.7.2
 + torchtnt==0.2.4
 + typing-inspect==0.9.0
 + wandb==0.25.1
 + websockets==16.0
 + werkzeug==3.1

In [5]:
# test with calculator 
from ase.build import bulk
from fairchem.core import pretrained_mlip, FAIRChemCalculator

atoms = bulk("Si")

model_name = "uma-s-1p1"
predictor = pretrained_mlip.get_predict_unit(model_name)
calc = FAIRChemCalculator(predictor, task_name="omat")

atoms.calc = calc
e = atoms.get_potential_energy()
print(e)

LocalEntryNotFoundError: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

In [ ]:

# pip install fairchem-core
from fairchem.core import pretrained_mlip, FAIRChemCalculator
from ase.io import read, write
from ase.build import molecule

model_name = "uma-s-1p1"
predictor = pretrained_mlip.get_predict_unit(model_name)
calc = FAIRChemCalculator(predictor, task_name="omat")


W0406 14:28:14.719000 98380 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


KeyError: "Model 'uma-odac' not found. Available models: ('uma-s-1', 'uma-s-1p1', 'uma-m-1p1', 'esen-md-direct-all-omol', 'esen-sm-conserving-all-omol', 'esen-sm-direct-all-omol', 'esen-sm-conserving-all-oc25', 'esen-md-direct-all-oc25')"

In [ ]:

# Optimize bare MOF
mof = read("mg_mof74.cif")
mof.calc = calc
opt_mof = BFGS(mof, logfile="mof_opt.log")
opt_mof.run(fmax=0.01)


E_mof = mof.get_potential_energy()
print(f"E(Mg-MOF-74) = {E_mof:.6f} eV")


# Optimize isolated CO2
co2 = molecule("CO2")
co2.calc = calc
opt_co2 = BFGS(co2, logfile="co2_opt.log")
opt_co2.run(fmax=0.01)


E_co2 = co2.get_potential_energy()
print(f"E(CO2) = {E_co2:.6f} eV")


# Optimize MOF + CO2
mof_co2 = read("mof_co2_initial.cif")
mof_co2.calc = calc
opt = BFGS(mof_co2, logfile="mof_co2_opt.log")
opt.run(fmax=0.01)


E_mof_co2 = mof_co2.get_potential_energy()
print(f"E(CO2@Mg-MOF-74) = {E_mof_co2:.6f} eV")


write("mof_co2_relaxed.cif", mof_co2)


# Adsorption energy
delta_E = E_mof_co2 - (E_mof + E_co2)
print(f"Adsorption energy: {delta_E:.4f} eV")


# Mg–CO2 distance
mg_indices = [i for i, atom in enumerate(mof_co2) if atom.symbol == "Mg"]
co2_indices = list(range(len(mof_co2)-3, len(mof_co2)))


for mg_i in mg_indices:
   for co2_j in co2_indices:
       d = mof_co2.get_distance(mg_i, co2_j, mic=True)
       if d < 3.5:
           print(f"Mg[{mg_i}] — {mof_co2[co2_j].symbol}[{co2_j}]: {d:.3f} Å")